# Function 8

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.0, 0.25, 0.0, 0.0, 0.0, 0.125, 0.0, 1.0], [0.0, 0.5, 0.0, 0.25, 0.5, 0.125, 0.0, 0.0], [0.0, 0.25, 0.0, 1.0, 0.25, 0.75, 0.25, 1.0]]
additionalOutputs = [np.float64(8.798300000000001), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.340175), np.float64(9.495175), np.float64(8.96205)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# --- Latin Hypercube Sampling (LHS) ---
n_samples = 9500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(9500000, 8)


In [5]:
print(x_grid[:10])

[[0.72632055 0.28865322 0.29146296 0.86507866 0.47087683 0.20075895
  0.49854392 0.93893002]
 [0.79118988 0.6237669  0.71152354 0.05663443 0.09139414 0.74466044
  0.8059629  0.56636756]
 [0.04249626 0.75989726 0.90904275 0.9133772  0.84705287 0.65444828
  0.61800653 0.54697106]
 [0.6211976  0.53276861 0.62969185 0.12658768 0.63255198 0.0203014
  0.30809266 0.86606032]
 [0.6206467  0.3574627  0.44247458 0.35911009 0.18975672 0.17961216
  0.8535585  0.95966456]
 [0.84469553 0.66486844 0.53313287 0.86484439 0.89956054 0.76162076
  0.63555943 0.33815029]
 [0.9608714  0.05163177 0.58999608 0.86258558 0.45001686 0.42980267
  0.15233614 0.02969287]
 [0.95986679 0.35011078 0.88203746 0.92106378 0.40406972 0.42952079
  0.97304899 0.58883866]
 [0.59737614 0.96151068 0.69768078 0.97757144 0.35414852 0.95244038
  0.66897661 0.63289133]
 [0.02906023 0.89187292 0.21042305 0.32890155 0.04088681 0.22212319
  0.30873836 0.61066423]]


# Bayesian Optimisation with UCB applied

In [6]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
#beta = 1.96
#beta = 0.5
beta = 0.02

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.2332018  0.61749524 0.44679553 0.49315594 0.78896169 0.61884089
 0.3071253  0.23723021]
